In [1]:
import pandas as pd
import numpy as np
import re
import os

# Cek apakah file dataset terbaca
# Kita mundur satu folder (..), lalu masuk ke DATASETUAP, lalu cari data.csv
path_dataset = '../DATASETUAP/data.csv'

if os.path.exists(path_dataset):
    print("✅ File dataset DITEMUKAN! Siap diproses.")
else:
    print("❌ File dataset TIDAK DITEMUKAN.")
    print(f"Pastikan nama file di folder DATASETUAP adalah 'data.csv'")

✅ File dataset DITEMUKAN! Siap diproses.


In [2]:
# Load Data
df = pd.read_csv(path_dataset, encoding='latin-1')
print(f"Jumlah data awal: {len(df)} baris")

# Tampilkan 5 data teratas untuk memastikan isinya benar
print("Contoh Data:")
display(df.head())

# ==============================================================================
# PROSES LABELING (Toxic vs Non-Toxic)
# ==============================================================================
def label_toxic(row):
    # Jika mengandung Hate Speech (HS) ATAU Abusive, maka Toxic (1)
    if row['HS'] == 1 or row['Abusive'] == 1:
        return 1
    else:
        return 0

# Terapkan fungsi labeling
df['label'] = df.apply(label_toxic, axis=1)

print("\nDistribusi Label:")
print(df['label'].value_counts())
print("(Keterangan: 1 = Toxic, 0 = Aman)")

Jumlah data awal: 13169 baris
Contoh Data:


,Tweet,HS,Abusive,HS_Individual,HS_Group,HS_Religion,HS_Race,HS_Physical,HS_Gender,HS_Other,HS_Weak,HS_Moderate,HS_Strong
0,- disaat semua cowok berusaha melacak perhatia...,1,1,1,0,0,0,0,0,1,1,0,0
1,RT USER: USER siapa yang telat ngasih tau elu?...,0,1,0,0,0,0,0,0,0,0,0,0
2,"41. Kadang aku berfikir, kenapa aku tetap perc...",0,0,0,0,0,0,0,0,0,0,0,0
3,USER USER AKU ITU AKU\n\nKU TAU MATAMU SIPIT T...,0,0,0,0,0,0,0,0,0,0,0,0
4,USER USER Kaum cebong kapir udah keliatan dong...,1,1,0,1,1,0,0,0,0,0,1,0



Distribusi Label:
label
1    7309
0    5860
Name: count, dtype: int64
(Keterangan: 1 = Toxic, 0 = Aman)


In [3]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Siapkan tools pembersih
# Note: Kita matikan stemming dulu biar cepat (karena data kamu 13.000 baris)
stop_factory = StopWordRemoverFactory()
stopword = stop_factory.create_stop_word_remover()

def clean_text(text):
    text = str(text).lower() # Lowercase
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # Hapus URL
    text = re.sub(r'@\w+','', text) # Hapus mention
    text = re.sub(r'[^a-z\s]', '', text) # Hapus simbol & angka
    text = re.sub(r'\s+', ' ', text).strip() # Hapus spasi dobel
    text = stopword.remove(text) # Hapus kata sambung (yang, di, ke..)
    return text

print("Sedang membersihkan data... (Tunggu sebentar, sekitar 10-30 detik)")
df['text_clean'] = df['Tweet'].apply(clean_text)

print("✅ Data berhasil dibersihkan!")
display(df[['text_clean', 'label']].head())

Sedang membersihkan data... (Tunggu sebentar, sekitar 10-30 detik)
✅ Data berhasil dibersihkan!


,text_clean,label
0,disaat semua cowok berusaha melacak perhatian ...,1
1,rt user user siapa telat ngasih tau eluedan sa...,1
2,kadang aku berfikir aku tetap percaya tuhan pa...,0
3,user user aku akunnku tau matamu sipit diliat ...,0
4,user user kaum cebong kapir udah keliatan dong...,1


In [4]:
# Hapus baris kosong (jaga-jaga)
df.dropna(subset=['text_clean'], inplace=True)

# Simpan ke folder yang sama
output_path = '../DATASETUAP/data_bersih.csv'
df[['text_clean', 'label']].to_csv(output_path, index=False)

print(f"✅ File bersih tersimpan di: {output_path}")
print("Siap lanjut ke tahap Training Model!")

✅ File bersih tersimpan di: ../DATASETUAP/data_bersih.csv
Siap lanjut ke tahap Training Model!
